# GIS to Unreal Engine 5.5 — Heightmap + Track Prep

### Dependencies

In [ ]:
import os, glob
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.io import MemoryFile
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.fill import fillnodata
import matplotlib.pyplot as plt
import pyproj
import gpxpy
import imageio
from scipy.ndimage import zoom
import contextily as ctx
from scipy.interpolate import RegularGridInterpolator

### GIS to Unreal Engine 5.5 Pre-processing
Mosaics DEM tiles, reprojects to UTM, tiles heightmaps (2017x2017, 16-bit) with 1px overlap, exports track as OBJ in centimeters.

In [2]:
# --- PARAMETERS ---
input_paths = glob.glob("../data/wurl/usgs_tiles/*.tif")
gpx_path = "../data/wurl/WURL_Wasatch_Ultimate_Ridge_Linkup.gpx"
output_dir = "../data/wurl/unreal_output"
tile_size = 4033

os.makedirs(output_dir, exist_ok=True)

# --- 1. LOAD GPX & DETERMINE UTM ZONE ---
with open(gpx_path) as f:
    gpx = gpxpy.parse(f)
points = []
for track in gpx.tracks:
    for segment in track.segments:
        for pt in segment.points:
            points.append({'lat': pt.latitude, 'lon': pt.longitude, 'ele': pt.elevation or 0})
if not points:
    raise ValueError("No track points found in GPX.")
print(f"Loaded GPX: {len(points)} points")

center_lon = np.mean([p['lon'] for p in points])
center_lat = np.mean([p['lat'] for p in points])
zone = int(np.floor((center_lon + 180) / 6)) + 1
hemisphere = 6 if center_lat >= 0 else 7
target_crs = f"EPSG:326{zone:02d}" if hemisphere == 6 else f"EPSG:327{zone:02d}"
print(f"UTM zone: {zone}{'N' if hemisphere == 6 else 'S'}  ({target_crs})")

# Reproject GPX to UTM meters
proj = pyproj.Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)
coords = np.array([proj.transform(p['lon'], p['lat']) for p in points])
elevs = np.array([p['ele'] for p in points])

E_min, E_max = coords[:, 0].min(), coords[:, 0].max()
N_min, N_max = coords[:, 1].min(), coords[:, 1].max()
print(f"Track bounds (UTM m): E [{E_min:.0f}, {E_max:.0f}], N [{N_min:.0f}, {N_max:.0f}]")

# --- 2. MOSAIC USGS TILES ---
opened = []
if len(input_paths) == 1:
    src = rasterio.open(input_paths[0])
    opened.append(src)
else:
    handles = [rasterio.open(p) for p in input_paths]
    mosaic_array, mosaic_transform = merge(handles)
    profile = handles[0].profile.copy()
    profile.update(height=mosaic_array.shape[1], width=mosaic_array.shape[2], transform=mosaic_transform)
    mem = MemoryFile()
    src = mem.open(**profile)
    src.write(mosaic_array)
    opened.extend(handles + [mem, src])

# --- 3. REPROJECT TO UTM ---
dst_crs = rasterio.crs.CRS.from_string(target_crs)
tfm, w_out, h_out = calculate_default_transform(src.crs, dst_crs, src.width, src.height, *src.bounds)
p = src.profile.copy(); p.update(crs=dst_crs, transform=tfm, width=w_out, height=h_out)
m = MemoryFile(); src_r = m.open(**p)
for i in range(1, src.count + 1):
    reproject(rasterio.band(src, i), rasterio.band(src_r, i),
              src_transform=src.transform, src_crs=src.crs,
              dst_transform=tfm, dst_crs=dst_crs, resampling=Resampling.bilinear)
opened += [m, src_r]
print(f"Reprojected to {target_crs}: {w_out}x{h_out}")

# CHANGE 1: Capture pixel size after reprojection for reference
pixel_size_m = abs(src_r.transform.a)

# --- 4. FIND GPX CENTROID AND SET TILE-ALIGNED WINDOW ---
inv = ~src_r.transform
gpx_cols = inv.a * coords[:, 0] + inv.c
gpx_rows = inv.e * coords[:, 1] + inv.f
centroid_col = gpx_cols.mean()
centroid_row = gpx_rows.mean()

stride = tile_size - 1
cmin_gpx = gpx_cols.min()
cmax_gpx = gpx_cols.max()
rmin_gpx = gpx_rows.min()
rmax_gpx = gpx_rows.max()
n_tiles_x = max(1, int(np.ceil((cmax_gpx - cmin_gpx - 1) / stride)))
n_tiles_y = max(1, int(np.ceil((rmax_gpx - rmin_gpx - 1) / stride)))
target_cols = n_tiles_x * stride + 1
target_rows = n_tiles_y * stride + 1

cmin = int(round(centroid_col - (target_cols - 1) / 2))
rmin = int(round(centroid_row - (target_rows - 1) / 2))
cmax = cmin + target_cols
rmax = rmin + target_rows

cmin = max(0, cmin)
rmin = max(0, rmin)
cmax = min(w_out, cmax)
rmax = min(h_out, rmax)
target_cols = cmax - cmin
target_rows = rmax - rmin
n_tiles_x = int(np.ceil((target_cols - 1) / stride))
n_tiles_y = int(np.ceil((target_rows - 1) / stride))
print(f"Target distribution structure: {n_tiles_x}x{n_tiles_y} grid layouts")

# --- 6. CROP, FILL NODATA, NORMALIZE ---
w = Window(cmin, rmin, target_cols, target_rows)
cropped = src_r.read(window=w).astype(np.float32)
crop_tfm = rasterio.windows.transform(w, src_r.transform)

# Fill NoData voids
nodata = src_r.nodata
if nodata is not None:
    mask = cropped[0] != nodata
    cropped[0] = fillnodata(cropped[0], mask)

# Pad to exact grid math if window was truncated at DEM edge
rows, cols = cropped.shape[1], cropped.shape[2]
target_cols_exact = n_tiles_x * stride + 1
target_rows_exact = n_tiles_y * stride + 1
need_pad_c = max(0, target_cols_exact - cols)
need_pad_r = max(0, target_rows_exact - rows)
if need_pad_c > 0 or need_pad_r > 0:
    cropped = np.pad(cropped, ((0, 0), (0, need_pad_r), (0, need_pad_c)), mode='edge')
    rows, cols = cropped.shape[1], cropped.shape[2]

# Normalize to 16-bit
dem_min = float(cropped[0].min())
dem_max = float(cropped[0].max())
dem_16 = ((cropped[0] - dem_min) / (dem_max - dem_min) * 65535).astype(np.uint16)
print(f"Elevation range: {dem_min:.1f}m to {dem_max:.1f}m -> 0-65535")

# Z-Scale calculation for Unreal import
elevation_range = dem_max - dem_min
unreal_z_scale = (elevation_range * 100) / 512
print(f"--> WHEN IMPORTING TO UE5, SET THE PNG 'SCALE X/Y' TO: {100 * pixel_size_m:.4f}")
print(f"--> WHEN IMPORTING TO UE5, SET THE LANDSCAPE 'SCALE Z' TO: {unreal_z_scale:.4f}")

# --- 7. EXPORT HEIGHTMAP & FLUSH DRAG-AND-DROP TRACK ---

# 1. Export the heightmap
hm_dir = os.path.join(output_dir, "heightmaps")
os.makedirs(hm_dir, exist_ok=True)
hm_path = os.path.join(hm_dir, "Heightmap.png")
imageio.v3.imwrite(hm_path, dem_16, plugin="pillow")
print(f"Heightmap exported: {hm_path} ({cols}x{rows} px, {dem_16.nbytes / 1e6:.1f} MB)")

# 2. Capture the scale factor: How many real-world meters is 1 pixel?
pixel_size_m = abs(src_r.transform.a)

# 3. Find the geographic center of the image window in meters
image_width_meters = pixel_size_m * cols
image_height_meters = abs(src_r.transform.e) * rows

map_center_easting = crop_tfm.c + (image_width_meters / 2.0)
map_center_northing = crop_tfm.f - (image_height_meters / 2.0)

# 4. Calculate raw distance from the center in meters (Standard Cartesian coordinates)
x_meters_raw = coords[:, 0] - map_center_easting
y_meters_raw = coords[:, 1] - map_center_northing

y_meters_flipped = -y_meters_raw

# 6. COMPRESS SCALE TO MATCH UNREAL UNITS (Scale = pixel_size * 100 cm)
x_unreal = (x_meters_raw / pixel_size_m) * 100 * pixel_size_m
y_unreal = (y_meters_flipped / pixel_size_m) * 100 * pixel_size_m

# 7. FORCE CONSTANT Z FOR BLUEPRINT SHRINK WRAPPING
z_unreal = np.full_like(x_unreal, 150000.0)

# Export to your CSV Data Table
import pandas as pd
df = pd.DataFrame({
    'Point_ID': range(len(x_unreal)),
    'X': x_unreal,
    'Y': y_unreal,
    'Z': z_unreal
})

csv_path = os.path.join(output_dir, "track_points.csv")
df.to_csv(csv_path, index=False)
print(f"CSV successfully exported to: {csv_path}")

Loaded GPX: 940 points
UTM zone: 12N  (EPSG:32612)
Track bounds (UTM m): E [432505, 449341], N [4486021, 4496203]
Reprojected to EPSG:32612: 9483x12225
Target distribution structure: 1x1 grid layouts
Elevation range: 1265.3m to 3567.9m -> 0-65535
--> WHEN IMPORTING TO UE5, SET THE PNG 'SCALE X/Y' TO: 2742.7062
--> WHEN IMPORTING TO UE5, SET THE LANDSCAPE 'SCALE Z' TO: 449.7220
Heightmap exported: ../data/wurl/unreal_output\heightmaps\Heightmap.png (4033x4033 px, 32.5 MB)
CSV successfully exported to: ../data/wurl/unreal_output\track_points.csv


In [3]:
# --- 1. FULL DEM GRID ---
stride = w.width - 1
downsample_factor = 16

w_macro = Window(0, 0, w_out, h_out)

col_ticks = np.arange(0, w_out, downsample_factor)
row_ticks = np.arange(0, h_out, downsample_factor)
if col_ticks[-1] != w_out - 1:
    col_ticks = np.append(col_ticks, w_out - 1)
if row_ticks[-1] != h_out - 1:
    row_ticks = np.append(row_ticks, h_out - 1)
cols_lr = len(col_ticks)
rows_lr = len(row_ticks)

# --- 2. SAMPLE & BUILD VALIDITY MASK ---
sample_dem = src_r.read(1).astype(np.float32)
if src_r.nodata is not None:
    valid_mask = sample_dem != src_r.nodata
else:
    valid_mask = np.ones_like(sample_dem, dtype=bool)

low_res_dem = sample_dem[row_ticks[:, None], col_ticks[None, :]]
low_res_valid = valid_mask[row_ticks[:, None], col_ticks[None, :]]

# --- 3. 10-PX SKIRT DIP (valid vertices only) ---
skirt_width_px = 10
dip_depth_m = 20.0
c0, c1 = w.col_off, w.col_off + stride
r0, r1 = w.row_off, w.row_off + stride

for r in range(rows_lr):
    for c in range(cols_lr):
        if not low_res_valid[r, c]:
            continue
        col, row = col_ticks[c], row_ticks[r]
        dc = max(c0 - col, col - c1)
        dr = max(r0 - row, row - r1)
        dp = max(dc, dr)
        if dp <= 0:
            inside = abs(dp)
            if inside <= skirt_width_px:
                t = 1.0 - (inside / skirt_width_px)
                low_res_dem[r, c] -= dip_depth_m * t

# --- 4. CONVERT TO UNREAL ---
cx = w.col_off + stride / 2.0
cy = w.row_off + stride / 2.0
x_pos = (col_ticks - cx) * pixel_size_m * 100.0
y_pos = (row_ticks - cy) * -pixel_size_m * 100.0
z_ref = (dem_min + dem_max) / 2.0
z_pos = (low_res_dem - z_ref) * 100.0
X, Y = np.meshgrid(x_pos, y_pos)
vertices = np.vstack([X.ravel(), Y.ravel(), z_pos.ravel()]).T

# --- 5. UVs & DONUT FACES (valid quads only) ---
u_vals = np.linspace(0.0, 1.0, cols_lr)
v_vals = np.linspace(1.0, 0.0, rows_lr)
U, V = np.meshgrid(u_vals, v_vals)
uvs = np.column_stack([U.ravel(), V.ravel()]).astype(np.float32)

faces = []
for r in range(rows_lr - 1):
    for c in range(cols_lr - 1):
        v0 = r * cols_lr + c
        v1 = v0 + 1
        v2 = (r + 1) * cols_lr + c
        v3 = v2 + 1

        # A. DROP if any corner is NoData (DEM edge waterfall cut)
        if not (low_res_valid[r, c] and low_res_valid[r, c+1]
                and low_res_valid[r+1, c] and low_res_valid[r+1, c+1]):
            continue

        col, row = col_ticks[c], row_ticks[r]
        dc = max(c0 - col, col - c1)
        dr = max(r0 - row, row - r1)

        # B. DROP if past the 10px skirt zone (deep inside core)
        if max(dc, dr) <= -skirt_width_px:
            continue

        faces.append([v0, v2, v1])
        faces.append([v1, v2, v3])

print(f"Mesh done. Faces: {len(faces)}")

# --- 6. EXPORT OBJ ---
mesh_dir = os.path.join(output_dir, "far_lands")
os.makedirs(mesh_dir, exist_ok=True)
obj_path = os.path.join(mesh_dir, "FarLands_Backdrop.obj")
with open(obj_path, 'w') as f:
    f.write('# Unreal Full DEM Backdrop Mesh\n')
    for v in vertices:
        f.write(f"v {v[0]:.4f} {v[1]:.4f} {v[2]:.4f}\n")
    for uv in uvs:
        f.write(f"vt {uv[0]:.6f} {uv[1]:.6f}\n")
    for face in faces:
        v0, v1, v2 = face[0] + 1, face[1] + 1, face[2] + 1
        f.write(f"f {v0}/{v0} {v1}/{v1} {v2}/{v2}\n")

print(f"--> Exported: {obj_path}")


Extracting full master DEM for smooth bilinear interpolation...


NameError: name 'RegularGridInterpolator' is not defined

In [ ]:
def export_satellite_texture(window_obj, filename, resolution_dpi=400):
    tfm = rasterio.windows.transform(window_obj, src_r.transform)
    width = window_obj.width
    height = window_obj.height

    xmin = tfm.c
    ymax = tfm.f
    xmax = xmin + (width * abs(tfm.a))
    ymin = ymax - (height * abs(tfm.e))

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.axis('off')

    print(f"Fetching ESRI satellite data for {filename}...")
    ctx.add_basemap(ax, crs=src_r.crs.to_string(), source=ctx.providers.Esri.WorldImagery)

    out_path = os.path.join(output_dir, filename)
    plt.savefig(out_path, bbox_inches='tight', pad_inches=0, dpi=resolution_dpi)
    plt.close()
    print(f"--> Saved texture: {out_path}")

export_satellite_texture(w, "Core_Satellite_Albedo.png", resolution_dpi=600)
export_satellite_texture(w_macro, "FarLands_Satellite_Albedo.png", resolution_dpi=300)